[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/purple/notebooks/purple_hdac1_feasibility.ipynb)

# Deciding whether HDAC1 is worth modelling

**Purple group · HIV**

The group's second target is HDAC1, an enzyme that keeps HIV-1 hidden and silent inside resting cells. Before spending days building a model for it, this notebook asks whether that is a sensible thing to do: which of the many ChEMBL targets called HDAC1 is the right one, and whether the data behind it is good enough to learn from.

## What you will do

- Search ChEMBL for every target named HDAC1 and work out which one to use
- Curate its activity data with the very same pipeline the group wrote for HIV-1
- Judge the data: do repeated measurements agree, and is the chemistry varied enough
- Train a quick model and check it on molecules it has never seen
- Decide, with reasons, whether HDAC1 is worth modelling

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "purple"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Which ChEMBL target is HDAC1?

HDAC1 is a histone deacetylase: it strips small acetyl tags off the proteins that DNA is wound
around, which packs the DNA tighter and switches genes off. HIV-1 uses this to hide. The virus
parks a copy of itself in the DNA of resting cells and lets it fall silent, where no drug can
reach it. Blocking HDAC1 loosens that packaging and wakes the virus up, so the drugs the patient
is already taking can find it. That is why it is interesting here as an **adjuvant**: not a
replacement for anti-HIV drugs, but something given alongside them.

HDAC1 never works alone. It sits inside large protein assemblies with names like Sin3, NuRD and
CoREST, and ChEMBL records several of those partnerships as targets in their own right. Searching
for "HDAC1" therefore returns many entries, of four different kinds:

- a **single protein** is the one protein on its own;
- a **protein family** groups isoforms an assay could not tell apart, for instance a measurement
  made against HDAC1 and HDAC2 together;
- a **protein complex** is two proteins measured as one unit;
- a **protein-protein interaction** is a measurement of the contact between two proteins, which
  is what a molecular glue or a PROTAC degrader is tested against.

Choosing the wrong one is the easiest mistake to make, so we look at all of them before deciding.

Ask ChEMBL for every target whose name mentions HDAC1.

In [ ]:
import json
import urllib.request

import pandas as pd

CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"


def chembl_get(path, **params):
    """Fetch one page from the ChEMBL web interface and return it as a dictionary."""
    query = "&".join(f"{key}={value}" for key, value in params.items())
    with urllib.request.urlopen(f"{CHEMBL}/{path}?{query}") as handle:
        return json.load(handle)

Now run the search itself.

In [ ]:
hits = chembl_get("target/search.json", q="HDAC1", limit=50)["targets"]
print(f"{len(hits)} targets in ChEMBL match the word HDAC1")

A name alone does not tell us which one to use. What decides it is how many measurements each
target actually holds, so we ask for the count of every one. This takes about twenty seconds.

In [ ]:
rows = []
for target in hits:
    counts = chembl_get("activity.json", target_chembl_id=target["target_chembl_id"], limit=1)
    rows.append({"target": target["target_chembl_id"], "name": target["pref_name"],
                 "organism": target["organism"], "kind": target["target_type"],
                 "activities": counts["page_meta"]["total_count"]})

targets = pd.DataFrame(rows).sort_values("activities", ascending=False, ignore_index=True)
targets.head(10)

One target dwarfs all the others: **CHEMBL325**, human histone deacetylase 1 as a single protein.
It holds around nineteen thousand measurements, while the complexes and the interactions hold a
few dozen between them, and even those are mostly degrader experiments rather than tests of
whether a molecule blocks the enzyme.

Two of the larger entries are tempting and should be turned down anyway. `Histone deacetylase`
(CHEMBL2093865) looks big, but it lumps all eleven human isoforms together, so a molecule in it
may have been measured against HDAC6 and never against HDAC1. `Histone deacetylase (HDAC1 and
HDAC2)` is ambiguous in the same way, on a smaller scale. Size is not the point; what the number
actually measures is.

Print what was set aside, so the decision is written down rather than just made.

In [ ]:
TARGET_ID = "CHEMBL325"

chosen = targets[targets.target == TARGET_ID].iloc[0]
aside = targets[targets.kind.isin(["PROTEIN COMPLEX", "PROTEIN-PROTEIN INTERACTION"])]
print(f"Chosen: {TARGET_ID}, {chosen['name']}, {chosen.activities:,} measurements")
print(f"Set aside: {len(aside)} complexes and interactions holding "
      f"{aside.activities.sum():,} measurements between them")
aside[["target", "name", "kind", "activities"]]

> **Exercise:** try the same search for a target your group cares about, by changing `q="HDAC1"`
> in the search above. How many entries come back, and how many of them would you actually use?

## 2. The measurements behind the target

Downloading nineteen thousand records takes a few minutes, so it was done once and the result
saved in the repository as `data/chembl325_hdac1.csv`. It is the plain output of the same
interface used above, with only the columns this notebook needs.

Load the measurements and see how many molecules and papers they come from.

In [ ]:
activities = pd.read_csv("data/chembl325_hdac1.csv")

print(f"{len(activities):,} measurements")
print(f"{activities.molecule_chembl_id.nunique():,} different molecules")
print(f"{activities.document_chembl_id.nunique():,} papers and patents, "
      f"published {int(activities.document_year.min())} to {int(activities.document_year.max())}")

Not every measurement is the same kind of number, and numbers of different kinds cannot be
compared with each other. Let us see what is in here.

In [ ]:
kinds = pd.DataFrame({"measurements": activities.standard_type.value_counts().head(8)})
kinds["usual unit"] = activities.groupby("standard_type").standard_units.agg(
    lambda column: column.value_counts().index[0] if column.notna().any() else "-")
kinds

**IC50** is the bulk of it: the concentration of a molecule needed to halve the enzyme's activity,
reported in nanomolar. **Ki** and **Kd** measure how tightly a molecule binds, also in nanomolar,
and are close enough to be used alongside it.

The rest cannot join them. `kon` and `k_off` are speeds, not concentrations, from a different
kind of experiment. `Inhibition` is a percentage measured at one single concentration, so a
molecule showing "40%" might be weak, or strong but tested at a low dose — the number does not
say. Keeping those would mean averaging things that are not comparable, so they are dropped.

## 3. Curate the data with the group's own pipeline

Nothing new is written in this section. `scripts/curation.py` is the module the HIV-1 curation
notebook uses, and every function below comes out of it unchanged. That is deliberate: if the
group's pipeline works on a second, quite different target without being edited, it is a real
pipeline rather than a script that happened to fit one dataset.

Keep the potency measurements in nanomolar, and drop the ones ChEMBL itself has flagged.

In [ ]:
from scripts import curation

usable = activities[
    activities.standard_type.isin(["IC50", "Ki", "Kd"])
    & activities.standard_units.eq("nM")
    & activities.canonical_smiles.notna()
    & activities.standard_value.notna()
].copy()

flagged = usable.data_validity_comment.notna()
print(f"{len(usable):,} potency measurements in nM")
print(f"{flagged.sum()} of them carry a warning from ChEMBL and are dropped: "
      f"{', '.join(usable.loc[flagged, 'data_validity_comment'].unique())}")
usable = usable[~flagged]

A measurement is either an exact value or a bound. `> 50000` means the molecule was tested and
did nothing up to that dose; `< 5` means it was too potent for the assay to pin down. Both are
informative, and the pipeline knows what to do with each, so we translate the signs into the form
it expects.

In [ ]:
usable["relation"] = (
    usable.standard_relation.fillna("=").str.strip("'\" ").map(curation.RELATIONS)
)
usable.relation.value_counts().rename({"=": "exact value",
                                       ">": "no effect up to this dose",
                                       "<": "more potent than this dose"})

Now the molecules. The same compound can be written as several different SMILES strings, and
ChEMBL often stores it as a salt. `curation.standardize_smiles` strips the salt away, rewrites
each molecule in one agreed form, and gives it an **InChIKey**: a 27-character code that is
identical for any two copies of the same molecule, however they were written down. This step
takes a minute or two.

In [ ]:
standard = curation.standardize_smiles(usable.canonical_smiles.unique())
print(f"{usable.canonical_smiles.nunique():,} different SMILES strings were written down")
print(f"{standard.inchikey.nunique():,} different molecules are actually behind them")

Join the cleaned structures back on and put every concentration on the pActivity scale.

In [ ]:
records = usable.merge(standard, on="canonical_smiles", how="inner")
records = records[records.standard_value > 0]
records["value_nm"] = records.standard_value.astype(float)
records["pactivity"] = curation.pactivity(records.value_nm)

print(f"{len(records):,} measurements of {records.inchikey.nunique():,} molecules")
records.groupby("standard_type").agg(measurements=("pactivity", "size"),
                                     molecules=("inchikey", "nunique"))

Finally, one row per molecule. Where a molecule was measured several times the median of its
exact values is taken, and molecules that only ever got a bound are rescued when that bound
already lands on the decided side of the cutoff — "no effect up to 50 micromolar" settles the
question at a 1 micromolar cutoff, while "no effect up to 100 nanomolar" settles nothing.

In [ ]:
CUTOFF_NM = 1000.0   # 1 micromolar, the same cutoff the HIV-1 notebooks use
THRESHOLD = curation.pactivity(CUTOFF_NM)   # the same cutoff written as a pActivity

exact = curation.summarise_replicates(records, ["inchikey"])
bounded = curation.decisive_bounds(records, ["inchikey"], CUTOFF_NM)
bounded = bounded[~bounded.index.isin(exact.index)]   # a real value always beats a bound

hdac1 = pd.concat([exact[["pactivity", "n_exact"]], bounded[["pactivity", "n_exact"]]])
hdac1["smiles"] = records.drop_duplicates("inchikey").set_index("inchikey").smiles
hdac1["active"] = (hdac1.pactivity >= THRESHOLD).astype(int)
hdac1 = hdac1.dropna(subset=["smiles"])

This is the funnel from raw download to a usable dataset. Every step threw something away.

In [ ]:
print(f"{len(activities):,} measurements downloaded from ChEMBL")
print(f"{len(records):,} potency measurements that can be compared")
print(f"{len(hdac1):,} molecules with a label, of which")
print(f"    {len(exact):,} have a measured value")
print(f"    {len(bounded):,} were rescued from a decisive bound")

## 4. Is the data good enough to learn from?

A big dataset is not automatically a good one. Three things decide whether a model can learn
from it: do repeated measurements of the same molecule agree, does the potency cover a real
range, and are the molecules varied enough that the model has something to generalise from.

Set up the plots, in the purple group's colour.

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

### 4.1 Do repeated measurements agree?

Many molecules were measured more than once, in different laboratories. The **spread** is the
distance between a molecule's highest and lowest value, in log units: a spread of 1 means the
two labs disagreed by a factor of ten. If most molecules disagreed that much, no model could do
better than the noise.

In [ ]:
repeated = exact[exact.n_exact > 1]
print(f"{len(repeated):,} molecules were measured more than once")
print(f"half of them agree to within {repeated.spread.median():.2f} log units")
print(f"{(repeated.spread > 1).mean():.0%} disagree by more than tenfold")

Most of the mass sits hard against zero, which is what we want to see.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(repeated.spread, bins=40, range=(0, 4), color=nc.purple)
stylia.label(ax, xlabel="Disagreement between repeats (log units)", ylabel="Molecules",
             title="Repeated measurements mostly agree")

### 4.2 Does the potency cover a real range?

pActivity turns a concentration into a number where bigger means more potent: 1 micromolar is 6,
1 nanomolar is 9. A dataset where everything lands on the same value has nothing to teach a
model.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(hdac1.pactivity, bins=60, range=(3, 11.5), color=nc.purple)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Potency runs from {hdac1.pactivity.min():.1f} to "
                   f"{hdac1.pactivity.max():.1f}, median {hdac1.pactivity.median():.2f}")

The dashed line is the cutoff that splits active from inactive. That cutoff is a choice, not a
fact, so it is worth seeing what other choices would have given before settling on one.

In [ ]:
rows = []
for cutoff_nm in [100.0, 1000.0, 10000.0]:
    rescued = curation.decisive_bounds(records, ["inchikey"], cutoff_nm)
    rescued = rescued[~rescued.index.isin(exact.index)]
    combined = pd.concat([exact[["pactivity"]], rescued[["pactivity"]]])
    labels = combined.pactivity >= curation.pactivity(cutoff_nm)
    rows.append({"cutoff": f"{cutoff_nm / 1000:g} uM", "molecules": len(combined),
                 "active": int(labels.sum()), "active share": f"{labels.mean():.0%}"})
pd.DataFrame(rows)

At 1 micromolar the two classes come out close to even. That is worth noticing, because the
group's HIV-1 dataset is badly lopsided at exactly the same cutoff. The cutoff did not change;
the chemistry people chose to publish did. HDAC1 has been studied by chemists optimising
inhibitors, who publish their failures alongside their successes, whereas HIV-1 whole-cell data
is dominated by screens where almost nothing works.

Draw the two classes at the chosen cutoff.

In [ ]:
counts = hdac1.active.value_counts().rename({0: "inactive", 1: "active"})
counts = counts.reindex(["inactive", "active"])

fig, axs = stylia.create_figure(1, 1, width=0.5)
ax = axs.next()
ax.bar(counts.index, counts.values, width=0.4, color=[nc.purple, nc.mint])
ax.set_xlim(-0.6, 1.6)
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

### 4.3 Is the chemistry varied enough?

A **Murcko scaffold** is what remains of a molecule once its decorations are stripped off and
only the ring skeleton is left. Counting scaffolds tells us whether this is a varied collection
or the same few series measured over and over.

In [ ]:
from scripts import modelling

scaffolds = pd.Series(modelling.murcko_scaffolds(hdac1.smiles.tolist()))
share = scaffolds.value_counts().head(10).sum() / len(scaffolds)
print(f"{scaffolds.nunique():,} different ring skeletons across {len(scaffolds):,} molecules")
print(f"the 10 commonest cover only {share:.0%} of the set")

There is one thing worth checking by hand. Nearly every known HDAC inhibitor works the same way:
a **hydroxamic acid** group reaches into the enzyme's pocket and grabs the zinc ion at the bottom
of it. If almost all the active molecules carry that group, a model could score well simply by
learning to spot it, and would be useless on anything else.

In [ ]:
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")   # RDKit warns about every molecule it cannot read
hydroxamic_acid = Chem.MolFromSmarts("[CX3](=O)[NX3][OX2H1]")
hdac1["hydroxamate"] = [
    bool(mol) and mol.HasSubstructMatch(hydroxamic_acid)
    for mol in (Chem.MolFromSmiles(smiles) for smiles in hdac1.smiles)
]
print(f"{hdac1.hydroxamate.mean():.0%} of the molecules contain a hydroxamic acid")
hdac1.groupby("hydroxamate").active.agg(molecules="size", active_share="mean").round(2)

Half of them do, and those are indeed more often active. But the other half are still active
about half the time, so the zinc binder is not the only thing there is to learn. Keep this in
mind: it comes back at the end.

## 5. Would a model learn anything?

This is the real test, and it is the same baseline the group trained for HIV-1: a random forest
on Morgan fingerprints. What matters is not the score itself but **how it is measured**.

A **random split** scatters molecules into folds at random, so a molecule's close relatives
usually sit in the training set and the model can half-remember the answer. A **scaffold split**
keeps every molecule sharing a ring skeleton together in one fold, so the model is tested on
chemistry it has genuinely never seen. The gap between the two is the honest number. A model
that only memorises series scores well on the first and collapses on the second.

Turn each molecule into a fingerprint: 2048 yes-or-no answers about what it contains.

In [ ]:
X = modelling.morgan_fingerprints(hdac1.smiles.tolist())
y = hdac1.active.values
print(f"{X.shape[0]:,} molecules, each described by {X.shape[1]} fingerprint bits")

Train and score the classifier both ways. This takes a couple of minutes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold

SEED = 42
forest = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                                class_weight="balanced")
random_split = modelling.cross_validate(
    forest, X, y, StratifiedKFold(5, shuffle=True, random_state=SEED))
scaffold_split = modelling.cross_validate(
    forest, X, y, GroupKFold(5), groups=scaffolds.values)

pd.DataFrame({"random split": random_split.mean(),
              "scaffold split": scaffold_split.mean()}).round(3)

Losing only a little when the split gets hard is the result we were hoping for. To be thorough,
try predicting the potency itself rather than the yes-or-no label, using only the molecules that
have a measured value. If the numbers can be predicted, the labels certainly can.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

measured = hdac1[hdac1.n_exact > 0]
X_measured = modelling.morgan_fingerprints(measured.smiles.tolist())
measured_scaffolds = pd.Series(modelling.murcko_scaffolds(measured.smiles.tolist()))
print(f"{len(measured):,} molecules have a measured pActivity")

Score the potency model on the same two splits.

In [ ]:
tree = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)

pd.DataFrame({
    "random split": modelling.cross_validate(
        tree, X_measured, measured.pactivity.values,
        KFold(5, shuffle=True, random_state=SEED)).mean(),
    "scaffold split": modelling.cross_validate(
        tree, X_measured, measured.pactivity.values,
        GroupKFold(5), groups=measured_scaffolds.values).mean(),
}).round(3)

> **Note:** an ROC-AUC of 0.5 would mean the model is guessing and 1.0 would mean it is never
> wrong. Around 0.9 on chemistry it has never seen is a genuinely useful model, and a much
> easier result than the same baseline reached on the group's HIV-1 data.

## 6. Is this really a second target?

One last check. If these were largely the same molecules the group already used for HIV-1, an
HDAC1 model would be a rerun on familiar chemistry rather than a new problem. Molecules are
compared by InChIKey, so the answer does not depend on how anyone wrote the SMILES.

Count the molecules the two datasets have in common.

In [ ]:
hiv1 = pd.read_csv("data/hiv1_curated.csv")
shared = set(hdac1.index) & set(hiv1.inchikey)

print(f"HIV-1 dataset: {len(hiv1):,} molecules")
print(f"HDAC1 dataset: {len(hdac1):,} molecules")
print(f"In both: {len(shared)} molecules, {len(shared) / len(hdac1):.2%} of the HDAC1 set")

A handful out of thousands. The two datasets are effectively separate collections of chemistry,
which is exactly what makes HDAC1 a fair test of whether the group's pipeline travels to a new
target rather than a second pass over the same molecules.

> **Exercise:** the molecules in both sets are worth a look. Which are they, and are they active
> against HIV-1, HDAC1, both, or neither? Use `hiv1[hiv1.inchikey.isin(shared)]`.

## Summary

- **CHEMBL325** is the target to use: human HDAC1 on its own. The complexes and protein-protein
  interactions sharing the name hold only a few dozen measurements between them, and those test
  degradation rather than inhibition.
- The group's HIV-1 curation pipeline ran on it **without a single change**, turning about 19,000
  raw measurements into roughly **9,700 labelled molecules**, close to evenly split at the 1
  micromolar cutoff.
- The data holds up to inspection: repeats agree closely, potency spans six orders of magnitude,
  and there are around 4,000 different ring skeletons.
- A baseline random forest reaches an **ROC-AUC near 0.9 on chemistry it has never seen**, only
  slightly below its score on the easy split. It is generalising, not memorising.
- Only a handful of molecules are shared with the HIV-1 dataset, so this is a genuine second
  target.

**The verdict: yes, HDAC1 is worth modelling.** The data is larger, cleaner and easier to learn
from than the group's HIV-1 data.

**One warning to carry forward.** Half of these molecules contain a hydroxamic acid, the group
that grabs the zinc ion, and the model has certainly learned to recognise it. African natural
products rarely contain one. So the model will be least confident exactly where the group most
wants to use it, and a good score here does not automatically transfer to a natural product
library.

**Next:** `purple_chemical_space.ipynb` shows how to map a set of molecules and see whether the
compounds you care about fall inside the region a model was trained on. Do that with the HDAC1
set and a natural product library before trusting any prediction on one.